# Example 11.4.2 / 2: Convex and Nonconvex MINLP

Companion notebook for **11.4.2 Mixed-Integer Nonlinear Models** in the AMPL lab Documentation, section "Convex and Nonconvex MINLP". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Convex and Nonconvex MINLP

A MINLP is convex when every subproblem left after fixing the integer variables is convex. Then
algorithms can use valid convex relaxations. Nonconvex terms need global relaxations and spatial
branching, which splits the range of a continuous variable into pieces and bounds each piece
separately, for a deterministic global guarantee; a local MINLP solver may return a feasible local
solution without proving the global optimum.

A one-binary nonconvex instance makes the difference visible. The objective
$(x^2-1)^2+0.1x+0.05\,\mathit{Use}$ has two wells. Bonmin, which treats the continuous
subproblems as convex, returns the right-hand well; Couenne, a global solver, returns the lower
left-hand well. Both report `solved`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Use binary;             # one binary
var x >= -2, <= 2, := 1.5;  # continuous, started in the right-hand well

minimize Wells: (x^2 - 1)^2 + 0.1 * x + 0.05 * Use;  # nonconvex: two wells

subject to Link: x >= -2 + Use;  # Use = 1 raises the lower bound on x

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for solver in ("bonmin", "couenne"):
    ampl.var["x"].set_value(1.5)
    ampl.solve(solver=solver, verbose=False)
    print(solver, ampl.solve_result, round(ampl.var["x"].value(), 6),
          round(ampl.obj["Wells"].value(), 6))

bonmin solved 0.987257 0.099367
couenne solved -1.012273 -0.100617
